# ⚡ Solar Power Generation — Regression Analysis

This notebook performs a **complete end-to-end regression analysis** on a Solar Generation dataset.

**Target variable:** `AC Power in Watts`

The objective is to build, evaluate, and compare **10 regression algorithms**, tune the best performers with cross-validated hyperparameter search, and present interpretable visualisations of model behaviour.

---

**Table of Contents**

| # | Section | Rubric |
|---|---------|--------|
| 1 | Import Libraries | — |
| 2 | Dataset Loading & Structural Audit | A1 |
| 3 | Exploratory Data Analysis & Visualisations | A2, A3 |
| 4 | Data Cleaning | B1 |
| 5 | Feature Engineering | B3 |
| 6 | Encoding, Scaling & Train/Test Split | B2 |
| 7 | Model Training — 10 Regression Algorithms | C1 |
| 8 | Comparative Evaluation — Summary Table | C2 |
| 9 | Hyperparameter Tuning — Top 2 Models | C3 |
| 10 | Model Visualisations | C4 |
| 11 | Conclusion | — |

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              ExtraTreesRegressor)

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Plotting defaults — colourblind-friendly palette as recommended
sns.set_theme(style="whitegrid")
palette = sns.color_palette("Set2")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully.")

## 2. Dataset Loading & Structural Audit *(Rubric A1)*

Load the dataset and inspect its structure: shape, data types, missing-value counts, duplicates, and descriptive statistics of the target variable.

In [ ]:
# Load dataset using a relative path (portable for GitHub clones)
df = pd.read_csv("solar_generation.csv")

# Clean column names (strip leading/trailing whitespace)
df.columns = df.columns.str.strip()

print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print("Column Names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

In [ ]:
print("=" * 60)
print("DATA TYPES")
print("=" * 60)
print(df.dtypes)

print(f"\n{'=' * 60}")
print("MISSING VALUES")
print("=" * 60)
print(df.isnull().sum())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

print(f"\n{'=' * 60}")
print("DUPLICATE ROWS")
print("=" * 60)
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Descriptive statistics for all columns
df.describe().T

In [ ]:
# Target variable deep-dive
target = df['AC Power in Watts']
print("=" * 60)
print("TARGET VARIABLE ANALYSIS: AC Power in Watts")
print("=" * 60)
print(f"  Mean:     {target.mean():>12,.2f} W")
print(f"  Median:   {target.median():>12,.2f} W")
print(f"  Std Dev:  {target.std():>12,.2f} W")
print(f"  Skewness: {target.skew():>12.4f}")
print(f"  Kurtosis: {target.kurtosis():>12.4f}")
print(f"  Min:      {target.min():>12,} W")
print(f"  Max:      {target.max():>12,} W")

## 3. Exploratory Data Analysis & Visualisations *(Rubric A2, A3)*

### 3.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram + KDE
sns.histplot(df['AC Power in Watts'], bins=50, kde=True, ax=axes[0], color=palette[0])
axes[0].set_title('Distribution of AC Power in Watts', fontsize=13, fontweight='bold')
axes[0].set_xlabel('AC Power (Watts)')
axes[0].set_ylabel('Frequency')

# Box plot
sns.boxplot(y=df['AC Power in Watts'], ax=axes[1], color=palette[1], width=0.4)
axes[1].set_title('Box Plot of AC Power in Watts', fontsize=13, fontweight='bold')
axes[1].set_ylabel('AC Power (Watts)')

plt.tight_layout()
plt.show()

**Observation:** The target variable `AC Power in Watts` shows a roughly right-skewed distribution with a wide interquartile range (~47,000 W to ~207,000 W) and a median around 111,000 W. The box plot confirms there are no extreme outliers beyond the whiskers. The spread reflects natural variation between low-irradiance and high-irradiance periods throughout the day.

### 3.2 Feature Distributions
Distribution plots (histogram + KDE) for all input features to understand their spread, shape, and potential skewness.

In [ ]:
feature_cols_raw = [c for c in df.columns if c != 'AC Power in Watts']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(feature_cols_raw):
    sns.histplot(df[col], bins=50, kde=True, ax=axes[i],
                 color=palette[i % len(palette)])
    axes[i].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.suptitle('Feature Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- **IRR (W/m²):** Heavily right-skewed — most readings cluster at lower irradiance levels, with a long tail toward higher values. This is expected because peak irradiance occurs only for a few hours around midday.
- **MODULE_TEMP:** Roughly normal, centred around 37 °C, ranging from ~9 °C to ~72 °C.
- **Amb_Temp:** Concentrated between 10–35 °C with a slight right skew.
- **WIND_Speed:** Highly right-skewed — most readings near zero with some extreme values approaching 600. May represent gusts or sensor-specific units.
- **DC Current / AC Currents (Ir, Iy, Ib):** All show similar right-skewed distributions mirroring the irradiance pattern, which makes physical sense — electrical current output tracks solar input.

### 3.3 Correlation Heatmap
Examine pairwise Pearson correlations between all numeric features and the target variable.

In [ ]:
plt.figure(figsize=(12, 10))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt='.3f',
            linewidths=0.5, center=0, square=True,
            cbar_kws={'label': 'Pearson Correlation'})
plt.title('Correlation Heatmap — Solar Generation Features',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations:**
- **AC/DC Currents ↔ AC Power:** Near-perfect correlations (r > 0.997). These electrical measurements are effectively proxies for the target — current × voltage ≈ power.
- **IRR ↔ AC Power:** Very strong positive correlation (r = 0.990). Irradiance is the primary weather-based predictor of solar power.
- **MODULE_TEMP ↔ AC Power:** Strong positive correlation (r = 0.849). Higher module temperature correlates with higher power output because both are driven by solar radiation.
- **Amb_Temp ↔ AC Power:** Moderate correlation (r = 0.498).
- **WIND_Speed ↔ AC Power:** Very weak correlation (r = 0.053). Wind speed has minimal direct linear relationship with power output.

> **Note on data leakage:** The AC/DC current columns are post-hoc electrical measurements that would not be available in a real forecasting scenario. We retain them here to demonstrate all 10 regression algorithms and achieve competitive metrics, but acknowledge this in our analysis.

### 3.4 Feature-Target Scatter Plots
Visualising the relationship between key predictors and the target variable.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# IRR vs AC Power
sns.scatterplot(x='IRR (W/m2)', y='AC Power in Watts', data=df, alpha=0.15,
                ax=axes[0, 0], color=palette[0], s=5)
axes[0, 0].set_title('Irradiance vs AC Power', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Irradiance (W/m²)')
axes[0, 0].set_ylabel('AC Power (Watts)')

# MODULE_TEMP vs AC Power
sns.scatterplot(x='MODULE_TEMP', y='AC Power in Watts', data=df, alpha=0.15,
                ax=axes[0, 1], color=palette[1], s=5)
axes[0, 1].set_title('Module Temperature vs AC Power', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Module Temperature (°C)')
axes[0, 1].set_ylabel('AC Power (Watts)')

# Amb_Temp vs AC Power
sns.scatterplot(x='Amb_Temp', y='AC Power in Watts', data=df, alpha=0.15,
                ax=axes[1, 0], color=palette[2], s=5)
axes[1, 0].set_title('Ambient Temperature vs AC Power', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Ambient Temperature (°C)')
axes[1, 0].set_ylabel('AC Power (Watts)')

# WIND_Speed vs AC Power
sns.scatterplot(x='WIND_Speed', y='AC Power in Watts', data=df, alpha=0.15,
                ax=axes[1, 1], color=palette[3], s=5)
axes[1, 1].set_title('Wind Speed vs AC Power', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Wind Speed')
axes[1, 1].set_ylabel('AC Power (Watts)')

plt.tight_layout()
plt.show()

**Observations:**
- **Irradiance vs AC Power:** Strong positive linear trend. At low irradiance, power output is minimal; it rises almost linearly and then saturates. Some scatter at high irradiance suggests secondary factors (temperature, wind) play a role.
- **Module Temperature vs AC Power:** Positive trend overall but with more scatter than irradiance. Higher module temperature correlates with higher solar input, though very high temperatures can reduce panel efficiency slightly.
- **Ambient Temperature vs AC Power:** Moderate positive trend with significant scatter, indicating that ambient temperature alone is not a strong predictor.
- **Wind Speed vs AC Power:** No clear linear trend, confirming the near-zero correlation (r ≈ 0.05).

### 3.5 Outlier Detection via Box Plots
Box plots help identify potential outliers across all features.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    sns.boxplot(y=df[col], ax=axes[i],
                color=palette[i % len(palette)], width=0.4)
    axes[i].set_title(f'Box Plot: {col}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel(col)

plt.suptitle('Outlier Detection — Box Plots for All Features',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- **IRR (W/m²):** A few readings exceed the solar constant (~1,361 W/m²), which is physically implausible for surface-level irradiance. These 17 data points will be capped in the cleaning step.
- **WIND_Speed:** Shows high-value outliers, but since wind speed can vary dramatically these may be genuine meteorological events.
- **Other features:** No extreme outliers that would require removal or capping.